In [15]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('expert_dataset.csv')
print(df.head)

<bound method NDFrame.head of               category      lat       lon  \
0      Known PFAS user  52.5041 -0.682637   
1      Known PFAS user  50.8719  6.034520   
2      Known PFAS user  50.2234  8.765600   
3      Known PFAS user  47.8586  5.347160   
4      Known PFAS user  39.4717 -0.538828   
...                ...      ...       ...   
44583            Known  51.2326  4.335670   
44584            Known  51.2302  4.337570   
44585            Known  51.3153  4.346180   
44586            Known  51.2343  4.334540   
44587            Known  51.2326  4.335070   

                                      name         city         country  \
0                                       3F        Corby  United Kingdom   
1                                       3M     Kerkrade     Netherlands   
2       3P - Performance Plastics Products       Karben         Germany   
3         3P Performance Plastics Products      Langres          France   
4      3P Productos Plásticos Performantes     Valenci

In [16]:
# Cleaning data 
df_clean = df.drop(columns=['pfas_sum', 'source_type', 'source_url', 'details', ])

In [17]:
# setting up the data
from sklearn.neighbors import KNeighborsRegressor
import numpy as np
import pandas as pd

# Columns to estimate
pfas_cols = ['pfos', 'pfoa', 'pfos_pfoa', 'pfna', 'pfbs', 'pfhxa', 'pfhxs']

# Choose matrix type
selected_matrix = "Groundwater"  
# Choose cordinate set
your_lat = 57.7381
your_lon = 10.5678

new_coords = np.array([[your_lat, your_lon]])


# Filter by matrix and drop rows missing required values
df_valid = df_clean[
    (df['matrix'] == selected_matrix)
].dropna(subset=['lat', 'lon'] + pfas_cols, how='any')

# Features: latitude and longitude
X = df_valid[['lat', 'lon']].values

# Targets: multiple PFAS columns
y = df_valid[pfas_cols].values



In [18]:
# remove outliers above the 95th percentile for each PFAS column
for col in pfas_cols:
    threshold = df[col].quantile(0.95)
    df = df[df[col] <= threshold]

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def split_data(X, y, test_size=0.2, random_state=42):
    """Splits features and targets (y should be log-transformed here)."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, shuffle=True
    )
    return X_train, X_test, y_train, y_test

def calculate_evaluation_metrics(y_true, y_pred, target_names):
    """Calculates R2 by comparing RAW (untransformed) values."""
    if y_true.shape != y_pred.shape:
        raise ValueError("y_true and y_pred must have the same shape.")
    
    r2_scores = [r2_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])]
    mean_concentrations = np.mean(y_true, axis=0)
    
    results_df = pd.DataFrame({
        'Target': target_names,
        'R2 Score': r2_scores,
        'Mean Concentration (y_true)': mean_concentrations
    })
    
    return results_df.set_index('Target')

def inverse_transform_predictions(y_pred_log):
    """Applies the inverse transformation: exp(x) - 1."""
    y_pred_raw = np.expm1(y_pred_log)
    y_pred_raw[y_pred_raw < 0] = 0 
    return y_pred_raw

def transform_targets(y_raw):
    """Applies a log-transformation: log(1 + x)."""
    y_transformed = np.log1p(y_raw)
    return y_transformed.values

def calculate_evaluation_metrics(y_true, y_pred, target_names):
    """Calculates R2, MAE, and RMSE by comparing RAW values."""
    if y_true.shape != y_pred.shape:
        raise ValueError("y_true and y_pred must have the same shape.")
    
    r2_scores = [r2_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])]
    maes = [mean_absolute_error(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])]
    rmses = [np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])) for i in range(y_true.shape[1])]
    mean_concentrations = np.mean(y_true, axis=0)
    
    results_df = pd.DataFrame({
        'Target': target_names,
        'R2 Score': r2_scores,
        'MAE (Raw Scale)': maes,
        'RMSE (Raw Scale)': rmses,
        'Mean Concentration (y_true)': mean_concentrations
    })
    
    return results_df.set_index('Target')

In [20]:
X_train, X_test, y_train, y_test = split_data(X, y)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# log-transform the target arrays
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [21]:
# Train one KNN model per PFAS target and evaluate jointly
knn_models = {}
y_pred_raw_list = []

for i, target in enumerate(pfas_cols):
    # train KNN on the i-th log-transformed target
    model = KNeighborsRegressor(n_neighbors=3, weights='distance')
    model.fit(X_train_scaled, y_train_log[:, i])
    knn_models[target] = model

    # predict (log), inverse-transform to raw, collect
    y_pred_log_i = model.predict(X_test_scaled)
    y_pred_raw_i = inverse_transform_predictions(y_pred_log_i)
    y_pred_raw_list.append(y_pred_raw_i)

# combine per-target predictions into shape (n_samples, n_targets)
y_pred_raw_all = np.column_stack(y_pred_raw_list)

# Evaluate across all targets (raw scale)
evaluation_results = calculate_evaluation_metrics(y_test, y_pred_raw_all, pfas_cols)
print("--- KNN (one model per target) EVALUATION RESULTS ---")
print(evaluation_results)

# Predict at new coordinates using each model
pred_new_list = []
for target in pfas_cols:
    pred_log = knn_models[target].predict(new_coords_scaled)
    pred_raw = inverse_transform_predictions(pred_log)
    pred_new_list.append(pred_raw.ravel()[0])  # scalar

pred_new = np.array(pred_new_list).reshape(1, -1)
print("\nPredicted PFAS at your coordinates (raw scale):")
print(pd.DataFrame(pred_new, columns=pfas_cols))


--- KNN (one model per target) EVALUATION RESULTS ---
           R2 Score  MAE (Raw Scale)  RMSE (Raw Scale)  \
Target                                                   
pfos       0.050870      2200.426909       8722.018302   
pfoa       0.348900      1116.261712       5237.262372   
pfos_pfoa  0.191937      3244.370867      10593.591068   
pfna       0.016674        38.062775        172.011890   
pfbs       0.047781      3283.778267      34519.097435   
pfhxa      0.219482      1513.455674       9709.433379   
pfhxs      0.432332      3032.422111      23569.460183   

           Mean Concentration (y_true)  
Target                                  
pfos                       2540.508129  
pfoa                       1635.784581  
pfos_pfoa                  4176.292710  
pfna                         42.951032  
pfbs                       3576.052323  
pfhxa                      1907.363613  
pfhxs                      3275.885871  

Predicted PFAS at your coordinates (raw scale):
     

In [22]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

param_grid = {
    'n_estimators': [100, 150, 200, 250, 300, 350, 400],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9]
}

gbr_models = {}
eval_rows = []
pred_new_list = []

for i, target in enumerate(pfas_cols):
    print(f"Training GBR for {target} ({i+1}/{len(pfas_cols)})")
    y_train_i = y_train_log[:, i]
    y_test_i = y_test_log[:, i]

    gbr = GradientBoostingRegressor(random_state=42)
    try:
        grid = GridSearchCV(gbr, param_grid=param_grid, scoring='r2', cv=5, n_jobs=-1, verbose=0)
        grid.fit(X_train_scaled, y_train_i)
        best_params = grid.best_params_
        print(f"  Best params: {best_params}")
    except Exception as e:
        print(f"  GridSearchCV failed for {target}, using defaults. Error: {e}")
        best_params = {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 3}

    final_gbr = GradientBoostingRegressor(**best_params, random_state=42)
    final_gbr.fit(X_train_scaled, y_train_i)
    gbr_models[target] = final_gbr

    # Predict on test set (log -> raw)
    y_pred_log = final_gbr.predict(X_test_scaled)
    y_pred_raw = inverse_transform_predictions(y_pred_log)
    y_test_raw = inverse_transform_predictions(y_test_i)

    r2 = r2_score(y_test_raw, y_pred_raw)
    mae = mean_absolute_error(y_test_raw, y_pred_raw)
    rmse = root_mean_squared_error(y_test_raw, y_pred_raw)

    eval_rows.append({
        'Target': target,
        'R2 Score': r2,
        'MAE (Raw Scale)': mae,
        'RMSE (Raw Scale)': rmse,
        'Mean Concentration (y_true)': np.mean(y_test_raw)
    })

    # Predict at the new coordinates
    pred_log_new = final_gbr.predict(new_coords_scaled)
    pred_raw_new = inverse_transform_predictions(pred_log_new)[0]
    pred_new_list.append(pred_raw_new)

# Compile results
gbr_evaluation_results = pd.DataFrame(eval_rows).set_index('Target')
pred_new_df = pd.DataFrame([pred_new_list], columns=pfas_cols)

print("\n--- GBR Evaluation Results (one model per PFAS) ---")
print(gbr_evaluation_results)
print("\nPredicted PFAS at your coordinates (raw scale):")
print(pred_new_df)

Training GBR for pfos (1/7)
  Best params: {'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 250}
Training GBR for pfoa (2/7)
  Best params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 150}
Training GBR for pfos_pfoa (3/7)
  Best params: {'learning_rate': 0.01, 'max_depth': 8, 'n_estimators': 300}
Training GBR for pfna (4/7)
  Best params: {'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 250}
Training GBR for pfbs (5/7)
  Best params: {'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 300}
Training GBR for pfhxa (6/7)
  Best params: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 150}
Training GBR for pfhxs (7/7)
  Best params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100}

--- GBR Evaluation Results (one model per PFAS) ---
           R2 Score  MAE (Raw Scale)  RMSE (Raw Scale)  \
Target                                                   
pfos      -0.027595      2376.911642       9075.387270   
pfoa       0.328590      1142.184400  